In [2]:
import pandas as pd
from factor_analyzer import FactorAnalyzer

bfi = pd.read_csv("dados/bfi.csv")
itens = [c for c in bfi.columns if c[0] in "ACENO" and c[1:].isdigit()]

# Itens de redação invertida: discordar deles indica mais do traço, não menos.
invertidos = ["A1", "C4", "C5", "E1", "E2", "O2", "O5"]

dados = bfi[itens].dropna().astype(int)
dados[invertidos] = 7 - dados[invertidos]

n_obs, p_itens = dados.shape

In [3]:
nomes_fatores = ["N", "E", "C", "A", "O"]

af = FactorAnalyzer(n_factors=5, rotation="varimax", method="ml")
af.fit(dados)

cargas = pd.DataFrame(af.loadings_, index=itens, columns=nomes_fatores)
comunalidades = pd.Series(af.get_communalities(), index=itens)

tabela = cargas.round(2).where(cargas.abs() >= 0.3, "")
tabela["Comunalidade"] = comunalidades.round(2)
tabela

,N,E,C,A,O,Comunalidade
A1,,,,0.39,,0.17
A2,,,,0.6,,0.42
A3,,,,0.66,,0.53
A4,,,,0.45,,0.31
A5,,0.35,,0.58,,0.49
C1,,,0.53,,,0.34
C2,,,0.62,,,0.43
C3,,,0.55,,,0.32
C4,,,0.65,,,0.49
C5,,,0.57,,,0.44


In [5]:
from scipy import stats
import numpy as np

R_bfi = np.corrcoef(dados.to_numpy(), rowvar=False)

def teste_rv(m):
    ajuste = FactorAnalyzer(n_factors=m, rotation=None, method="ml")
    ajuste.fit(dados)
    L_m = ajuste.loadings_
    Sigma_m = L_m @ L_m.T + np.diag(ajuste.get_uniquenesses())
    log_razao = np.linalg.slogdet(Sigma_m)[1] - np.linalg.slogdet(R_bfi)[1]
    gl = ((p_itens - m) ** 2 - (p_itens + m)) / 2
    qui2 = (n_obs - 1 - (2 * p_itens + 4 * m + 5) / 6) * log_razao
    return gl, qui2, stats.chi2.sf(qui2, gl)

testes_bfi = pd.DataFrame(
    [teste_rv(m) for m in range(5, 12)],
    index=pd.Index(range(5, 12), name="m"),
    columns=["gl", "qui-quadrado", "p-valor"],
)
testes_bfi.assign(
    gl=testes_bfi["gl"].astype(int),
    **{"qui-quadrado": testes_bfi["qui-quadrado"].round(1)},
    **{"p-valor": testes_bfi["p-valor"].map("{:.1e}".format)},
)

,gl,qui-quadrado,p-valor
m,,,
5,185,1490.6,1.2e-202
6,165,896.7,5.7e-101
7,146,619.2,1.6e-59
8,128,438.2,1.5e-35
9,111,316.1,1.4e-21
10,95,224.8,1.6e-12
11,80,147.7,6.2e-06


In [6]:
L_bfi = af.loadings_
residuos_bfi = R_bfi - (L_bfi @ L_bfi.T + np.diag(af.get_uniquenesses()))
fora_diagonal = residuos_bfi[~np.eye(p_itens, dtype=bool)]

rmsr = np.sqrt((fora_diagonal ** 2).mean())
prop_grandes = (np.abs(fora_diagonal) > 0.05).mean()
print(f"Raiz do erro quadrático médio dos resíduos: {rmsr:.3f}")
print(f"Proporção de resíduos acima de 0,05 em módulo: {prop_grandes:.0%}")

Raiz do erro quadrático médio dos resíduos: 0.029
Proporção de resíduos acima de 0,05 em módulo: 8%
